# Run ESFS on Sycon dataset
## Set up workspace
Import modules and set up workspace

In [ ]:
# import modules
import esfs
import plotly.express as px
from scipy.sparse import csc_matrix
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import FuncNorm
from matplotlib.cm import get_cmap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.stats import pearsonr
from matplotlib import colors as mcolors
from matplotlib.colors import to_rgba
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap, Normalize
from  matplotlib.cm import ScalarMappable
import math, os

# set working dir
os.chdir("/data/evassvis/fn76/sycon/sycon_clusterAnnotation/sycon_cluster_analyses")

In [ ]:
# set python default discrete class colour palette to one with more colours
Colours = px.colors.qualitative.Dark24
Colours.remove('#222A2A')
plt.rcParams['axes.prop_cycle'] = plt.cycler(color = Colours)
print(Colours)

## Process Sycon whole dataset

In [ ]:
# read in Sycon h5ad file
Sycon = ad.read_h5ad("04_preprocessed_scRNAseqs/Scil_cellFiltered.h5ad")

# create scaled matrix
Sycon = esfs.create_scaled_matrix(Sycon)

# calculate standard ESSs and EPs matrices
Sycon = esfs.parallel_calc_es_matrices(Sycon,
                                       secondary_features_label = "Self",
                                       save_matrices = np.array(["ESSs","EPs"]),
                                       use_cores = -1)

### Get genes responsible for batch effect and HVGs

In [ ]:
# compute a sparse matrix to store information about batches per cell
Sycon.obsm["orig.ident"] = csc_matrix(pd.get_dummies(Sycon.obs["orig.ident"]))

# calculate matrices, considering different batches
Sycon = esfs.parallel_calc_es_matrices(Sycon,
                                       secondary_features_label = "orig.ident",
                                       save_matrices = np.array(["ESSs","EPs", "SGs"]),
                                       use_cores = -1)

# get genes that are driving batch effect
Batch_Effect_Genes = np.asarray(Sycon.var_names[np.where(Sycon.varm['orig.ident_ESSs'] > 0.01)[0]])

In [ ]:
# get known important genes from seurat HVGs
top5_marker_clusters = pd.read_csv("07_notableGenes_clusterAnnotation/top5_marker_perCluster.tsv",
                                   sep = "\t")
Known_important_genes = np.asarray(top5_marker_clusters.gene)

### Rank genes

In [ ]:
# rank genes, excluding those contributing to batch effect and accounting for HVGs
Sycon = esfs.ES_rank_genes(Sycon, known_important_genes = Known_important_genes, exclude_genes = Batch_Effect_Genes)

In [ ]:
# save processed object to file
Sycon.write_h5ad(filename = "14_ESFS/Sycon_ESranked.h5ad")

### Plot gene clusters

In [ ]:
# if Sycon does not exist as an object (i.e., resuming a previous analysis), load it 
if "Sycon" not in locals():
    Sycon = ad.read_h5ad("14_ESFS/Sycon_ESranked.h5ad")

top5_marker_clusters = pd.read_csv("07_notableGenes_clusterAnnotation/top5_marker_perCluster.tsv",
                                       sep = "\t")
Known_important_genes = np.asarray(top5_marker_clusters.gene)

In [ ]:
# plot UMAPs of various top genes to choose the best one
Num_Top_Ranked_Genes = [1000,2000,3000,4000,8000]

for num_genes in Num_Top_Ranked_Genes:
    Top_ESS_Genes, Gene_Clust_Labels, Gene_Embedding = esfs.plot_top_ranked_genes_UMAP(Sycon,
                                                                                    num_genes,
                                                                                    clustering = "hdbscan",
                                                                                    known_important_genes = Known_important_genes)
    plt.savefig(f"14_ESFS/01_plots/top{num_genes}genes_UMAP.png", bbox_inches = "tight")
    plt.savefig(f"14_ESFS/01_plots/top{num_genes}genes_UMAP.pdf", bbox_inches = "tight")
    plt.close()

In [ ]:
# 3000 is the best one
Num_Top_Ranked_Genes = 3000
                        
Top_ESS_Genes, Gene_Clust_Labels, Gene_Embedding = esfs.plot_top_ranked_genes_UMAP(Sycon,
                                                                                   Num_Top_Ranked_Genes,
                                                                                   clustering = "hdbscan",
                                                                                   known_important_genes = Known_important_genes)

### Get cell UMAP from gene clusters

In [ ]:
Gene_Cluster_Embeddings, Gene_Cluster_Selected_Genes = esfs.get_gene_cluster_cell_UMAPs(Sycon,
                                                                                        Gene_Clust_Labels,
                                                                                        Top_ESS_Genes,
                                                                                        n_neighbors = 50,
                                                                                        min_dist = 0.1,
                                                                                        log_transformed = True)

In [ ]:
# update Sycon metadata with a new column with "Clusters [num_cluster]"
# this is because otherwise ESFS will complain, as "seurat_clusters" is numerical
Sycon.obs["seurat_clusters_full"] = "Cluster " + Sycon.obs["seurat_clusters"].astype(str)

In [ ]:
# plot cell UMAPs
esfs.plot_gene_cluster_cell_UMAPs(Sycon,
                                  Gene_Cluster_Embeddings,
                                  Gene_Cluster_Selected_Genes,
                                  cell_label = "seurat_clusters_full",
                                  ncol = 3, figsize = (16, 10),
                                  marker_size = 2, log2_gene_expression = True)
plt.savefig(f"14_ESFS/01_plots/top{Num_Top_Ranked_Genes}genes_UMAPs_byGeneClusters.png", bbox_inches = "tight")
plt.savefig(f"14_ESFS/01_plots/top{Num_Top_Ranked_Genes}genes_UMAPs_byGeneClusters.pdf", bbox_inches = "tight")

In [ ]:
Cell_Label = "g4792"

# plot cell UMAPs with specific genes
esfs.plot_gene_cluster_cell_UMAPs(Sycon,
                                  Gene_Cluster_Embeddings,
                                  Gene_Cluster_Selected_Genes,
                                  cell_label = Cell_Label,
                                  ncol = 3,
                                  figsize = (16, 10),
                                  marker_size = 2,
                                  log2_gene_expression = True)

plt.savefig(f"14_ESFS/01_plots/top{Num_Top_Ranked_Genes}genes_UMAPs_byGeneClusters_{Cell_Label}.png", bbox_inches = "tight")
plt.savefig(f"14_ESFS/01_plots/top{Num_Top_Ranked_Genes}genes_UMAPs_byGeneClusters_{Cell_Label}.pdf", bbox_inches = "tight")

In [ ]:
# plot the cell UMAP after single gene clusters (gene cluster 0)
Gene_Cluster_Embeddings, Gene_Cluster_Selected_Genes = esfs.get_gene_cluster_cell_UMAPs(Sycon,
                                                                                        Gene_Clust_Labels,
                                                                                        Top_ESS_Genes,
                                                                                        specific_cluster = [0],
                                                                                        n_neighbors = 50,
                                                                                        min_dist = 0.1,
                                                                                        log_transformed = True)

esfs.plot_gene_cluster_cell_UMAPs(Sycon,
                                  Gene_Cluster_Embeddings,
                                  Gene_Cluster_Selected_Genes,
                                  cell_label = "seurat_clusters_full",
                                  ncol = 1, figsize = (8, 5),
                                  marker_size = 2, log2_gene_expression = True)

plt.savefig("14_ESFS/01_plots/UMAPs_byGeneCluster0.png", bbox_inches = "tight")
plt.savefig("14_ESFS/01_plots/UMAPs_byGeneCluster0.pdf", bbox_inches = "tight")

In [ ]:
# plot the cell UMAP after single gene clusters (gene cluster 2)
Gene_Cluster_Embeddings, Gene_Cluster_Selected_Genes = esfs.get_gene_cluster_cell_UMAPs(Sycon,
                                                                                        Gene_Clust_Labels,
                                                                                        Top_ESS_Genes,
                                                                                        specific_cluster = [2],
                                                                                        n_neighbors = 50,
                                                                                        min_dist = 0.1,
                                                                                        log_transformed = True)

esfs.plot_gene_cluster_cell_UMAPs(Sycon,
                                  Gene_Cluster_Embeddings,
                                  Gene_Cluster_Selected_Genes,
                                  cell_label = "seurat_clusters_full",
                                  ncol = 1, figsize = (8, 5),
                                  marker_size = 1, log2_gene_expression = True)

plt.savefig("14_ESFS/01_plots/UMAPs_byGeneCluster2.png", bbox_inches = "tight")
plt.savefig("14_ESFS/01_plots/UMAPs_byGeneCluster2.pdf", bbox_inches = "tight")